In [1]:
import sys
import json
from tqdm import tqdm

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'

### Extract

In [2]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [3]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))

In [4]:
agent = AgentConnector.open()

In [5]:
agent.generate("Сколько будет 2 + 2?")

'2 + 2 = 4'

In [6]:
extractor = LLMExtractor(agent_conn=agent)

In [8]:
extracted_triplets = []

In [7]:
out = extractor.extract_thesises(raw_texts[0])

TEXT: Gregory: "Really ? why I use IQOO9 for so many days that I feel very fragrant ."
Alan: "If you really listen to digital bloggers , you can only buy that kind of cost - effective machine , but does everyone need it ? Does everyone play Yuanshen ?"
Kyle: "Ah , is there still a cost -effective machine for the recent machine ? Is not all the hardware shrinks more than last year , the price rises ? [ laugh haha ] ."
Alan: "The sub -brand is basically cost - effective ( the pile last year ) ."
Deborah: "9 with DC is definitely fragrant but you try again in summer ."
Gregory: "In summer , 865 , 870 and A13 A14 are all fever [ sweat ] ."
Ann: "What if the fever ? at least the game experience has stabilized at high frame rate , 90 or even 120 , TM 's 888 and 8gen1 is barely 60Hz , and it 's even hotter than others"
Amanda: "Your IQOO9 scheduling is quite conservative . My colleague 's Xiaomi Mi 12Pro is unhot every day . The manufacturers are conservatively scheduled . When running for a 

In [10]:
'   - gregory uses iqoo9 for many days'.strip('.-*')

'   - gregory uses iqoo9 for many days'

In [8]:
out

[[{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop': {'type': 'simple'}},
  {'name': '   - gregory uses iqoo9 for many days',
   'type': 'hyper',
   'prop': {}}],
 [{'name': 'using', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop': {'type': 'simple'}},
  {'name': '   - gregory uses iqoo9 for many days',
   'type': 'hyper',
   'prop': {}}],
 [{'name': 'iqoo9', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop': {'type': 'simple'}},
  {'name': '   - gregory uses iqoo9 for many days',
   'type': 'hyper',
   'prop': {}}],
 [{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop': {'type': 'simple'}},
  {'name': ' gregory feels very fragrant', 'type': 'hyper', 'prop': {}}],
 [{'name': 'feeling', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop': {'type': 'simple'}},
  {'name': ' gregory feels very fragrant', 'type': 'hyper', 'prop': {}}],
 [{'name': 'fragrant', 'type': 'object', 'prop': {}},
  {'name': 'hyper', 'prop

In [10]:
out

[]

In [11]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

 29%|██▉       | 1007/3483 [12:38:13<31:04:17, 45.18s/it]


KeyboardInterrupt: 

In [12]:
with open("tmp_extracted_triplets.json", 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_triplets, ensure_ascii=False))

### Update

In [ ]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", default_db="diaasq"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

In [ ]:
ids = kg_model.graph_db.create_triplets(extracted_triplets)
prepared_triplets = MemPipeline.match_triplets_by_id(extracted_triplets, ids)
kg_model.embeddings_db.add_triplets(prepared_triplets)

In [ ]:
kg_model.graph_db.close()